#**transformer architecture**

**Input Embedding:** Converts tokens into dense vectors and adds positional encoding for sequential information.

**Multi-Head Attention:** Allows the model to focus on different parts of the input simultaneously, calculating attention scores using queries, keys, and values.

**Feed-Forward Layer:** A fully connected network applied to each position independently, adding non-linearity and feature transformation.

**Linear & Softmax Layers:** Linear layers project features to the required dimension; softmax converts logits into probabilities for output predictions.

**Output Embedding:** Maps final hidden states back to the vocabulary space to generate the output sequence.



#**RAG**

RAG is an AI technique that combines information retrieval + text generation.

#**Working:**

 **Retrieve**→ Finds relevant information from a document/database.

 **Augment** → Adds the retrieved information to the prompt.

 **Generate** → LLM generates an answer using that information.

**Simple Flow:**
User Query → Retrieve Documents → Add Context → LLM → Answer

**Example:**
If you upload a PDF and ask “What is the main topic?”, RAG searches the PDF for relevant content and gives that content to the LLM to generate the answer.

**LangChain** is a framework used to build applications powered by Large Language Models (LLMs).

**langchain_groq** is an integration that allows LangChain to use Groq's fast LLM APIs.
Purpose: Connect LangChain → Groq → LLM.

**PyPDF** is a Python library used to read, extract, and manipulate PDF files.

**langchain_huggingface** is a LangChain integration package that connects Hugging Face models and embeddings with LangChain applications.


In [ ]:
!pip install langchain langchain_community langchain_text_splitters langchain_groq langchain_huggingface faiss-cpu pypdf --quiet

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# --------------------------------------------------
# STEP 1: Load PDF
# --------------------------------------------------

loader = PyPDFLoader(
    "Cat vs Dog Image Classification, Final Capstone Project.pdf"
)
documents = loader.load()


In [ ]:
# --------------------------------------------------
# STEP 2: Split into chunks
# --------------------------------------------------

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")

In [ ]:
# --------------------------------------------------
# STEP 3: Load Environment & Create Embeddings
# --------------------------------------------------

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},      # Use "cuda" if GPU is available
    encode_kwargs={"normalize_embeddings": True}
)


vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

# Optional: Save locally for persistence
vectorstore.save_local("pdf_faiss_index")

print(" PDF indexed into FAISS vector database")

In [ ]:
# --------------------------------------------------
# STEP 5: Create Retriever
# --------------------------------------------------

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)


In [ ]:
query = "renewable energy usage across offices"

results = retriever.invoke(query)

print(f"Retrieved {len(results)} documents\n")

for i, doc in enumerate(results, 1):
    print(f"Document {i}")
    print("-" * 50)
    print(doc.page_content)
    print("\nMetadata:", doc.metadata)
    print()

In [ ]:
from google.colab import userdata
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

In [ ]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=GROQ_API_KEY,
    temperature=0.2,
)

In [ ]:
response = llm.invoke("Explain Retrieval-Augmented Generation in one paragraph.")
print(response.content)

In [ ]:
import gradio as gr

In [ ]:

# --------------------------------------------------
# Ask Question
# --------------------------------------------------

def ask_question(query):

    if query.strip() == "":
        return "", ""

    # Retrieve documents
    docs = retriever.invoke(query)

    # Build context
    context = ""

    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get("page", "N/A")
        source = doc.metadata.get("source", "Unknown")

        context += (
            f"[Source {i} - File: {source}, Page: {page}]\n"
            f"{doc.page_content}\n\n"
        )

    # Create prompt
    prompt = f"""
Answer the question using only the context below.

If the answer is not present, say:
"I could not find the answer in the provided document."

Context:
{context}

Question:
{query}

Answer:
"""

    # Call LLM
    response = llm.invoke(prompt)

    answer = response.content

    # Format retrieved sources
    source_text = ""

    for i, doc in enumerate(docs, 1):

        source_text += f"Source {i}\n"
        source_text += f"File : {doc.metadata.get('source','Unknown')}\n"
        source_text += f"Page : {doc.metadata.get('page','N/A')}\n"
        source_text += f"Content :\n{doc.page_content[:300]}...\n"
        source_text += "-" * 60 + "\n"

    return answer, source_text


In [ ]:



# --------------------------------------------------
# Gradio UI
# --------------------------------------------------

demo = gr.Interface(
    fn=ask_question,
    inputs=gr.Textbox(
        label="Question",
        placeholder="Ask something about the PDF..."
    ),
    outputs=[
        gr.Textbox(label="Answer", lines=8),
        gr.Textbox(label="Retrieved Sources", lines=15)
    ],
    title="PDF RAG Chatbot"
)

demo.launch()